# Assignment 3 Code Notebook

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend',  fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## Importing the data from kaggle

In [ ]:
from pathlib import Path
import kagglehub

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
print("Path to dataset files:", path)

df = pd.read_csv(Path(path) / "WA_Fn-UseC_-Telco-Customer-Churn.csv")

## Info about the data frame

In [ ]:
df.info()

In [ ]:
df[['tenure', 'TotalCharges', 'MonthlyCharges', 'Contract', 'PaymentMethod']].head(20)

In [ ]:
# TotalCharges is missing from this summary because it was loaded as text, not numbers
df.describe()

In [ ]:
df['Contract'].value_counts()

In [ ]:
df['PaymentMethod'].value_counts()

In [ ]:
df['Churn'].value_counts()

## Finding the missing values in TotalCharges

In [ ]:
# This says no column has missing values, but that isn't true:
# the blanks in TotalCharges are spaces (" "), which count as normal text
print(df.isnull().sum())

In [ ]:
# Changing the values to numbers. The whitespace can't be converted,
# so errors='coerce' turns it into NaN (a real missing value)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# This shows the true result. TotalCharges is the only column with missing values
print(df.isnull().sum())

In [ ]:
# All the missing rows have tenure 0 (new customers who haven't been billed yet)
df[df['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges']]

## Dropping irrelevant columns
I removed gender and phone service as well because they will just act as noise

In [ ]:
X = df.drop(columns=['customerID', 'gender', 'PhoneService', 'Churn'])
df['ChurnFlag'] = (df["Churn"] == "Yes")
y = df['ChurnFlag']

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

print(num_cols)
print(cat_cols)

## Pipelines for Cat and Num data frames

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# The median imputer fills the missing TotalCharges values
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("cat_encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess_pipe = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

In [ ]:
from sklearn.linear_model import SGDClassifier

final_pipe = Pipeline([
    ("preprocess", preprocess_pipe),
    ("sgd_clf", SGDClassifier(loss="log_loss", random_state=42))
])
final_pipe

## Training and evaluating on 3 train/test splits

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.metrics import (confusion_matrix, classification_report, precision_score,
                            recall_score, f1_score, roc_curve, roc_auc_score)

test_sizes = (0.2, 0.3, 0.4)
split_names = ("80/20", "70/30", "60/40")  # train/test

yTests, yPreds, yScoresAll = [], [], []

for test_size, split_name in zip(test_sizes, split_names):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size,
                                                        random_state=42)
    model = clone(final_pipe)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_scores = model.predict_proba(X_test)[:, 1]  # probability of churning

    yTests.append(y_test)
    yPreds.append(y_pred)
    yScoresAll.append(y_scores)

    print(f"===== Split {split_name}: {len(X_train)} train, {len(X_test)} test =====")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15, 4))
plt.rc('font', size=10)
for idx, split_name in enumerate(split_names):
    ConfusionMatrixDisplay.from_predictions(yTests[idx], yPreds[idx], ax=axs[idx])
    axs[idx].set_title(f"Split {split_name}")
plt.show()
plt.rc('font', size=14)

In [ ]:
plt.figure(figsize=(6, 5))
for idx, split_name in enumerate(split_names):
    fpr, tpr, thresholds = roc_curve(yTests[idx], yScoresAll[idx])
    auc = roc_auc_score(yTests[idx], yScoresAll[idx])
    plt.plot(fpr, tpr, linewidth=2, label=f"{split_name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], 'k:', label="Random classifier's ROC curve")
plt.xlabel('False Positive Rate (Fall-Out)')
plt.ylabel('True Positive Rate (Recall)')
plt.grid()
plt.axis([0, 1, 0, 1])
plt.legend(loc="lower right", fontsize=11)
plt.show()

In [ ]:
for idx, split_name in enumerate(split_names):
    yTest, yPred, yScores = yTests[idx], yPreds[idx], yScoresAll[idx]
    print(f"{split_name}  "
          f"accuracy: {(yPred == yTest).mean():.2%}  "
          f"precision: {precision_score(yTest, yPred):.2%}  "
          f"recall: {recall_score(yTest, yPred):.2%}  "
          f"f1: {f1_score(yTest, yPred):.2%}  "
          f"ROC-AUC: {roc_auc_score(yTest, yScores):.3f}")

## Comparing StandardScaler with MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax_num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler()),
])
minmax_pipe = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", minmax_num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])),
    ("sgd_clf", SGDClassifier(loss="log_loss", random_state=42))
])

for test_size, split_name in zip(test_sizes, split_names):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size,
                                                        random_state=42)
    model = clone(minmax_pipe)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_scores = model.predict_proba(X_test)[:, 1]
    print(f"MinMaxScaler {split_name}  "
          f"accuracy: {(y_pred == y_test).mean():.2%}  "
          f"recall: {recall_score(y_test, y_pred):.2%}  "
          f"ROC-AUC: {roc_auc_score(y_test, y_scores):.3f}")

## 2D decision boundary (tenure vs MonthlyCharges)

In [ ]:
X_2d = df[["tenure", "MonthlyCharges"]].values
y_2d = df["ChurnFlag"].values
X_train, X_test, y_train, y_test = train_test_split(X_2d, y_2d, test_size=0.2,
                                                    random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

sgd_2d = SGDClassifier(loss="log_loss", random_state=42)
sgd_2d.fit(X_train_scaled, y_train)

In [ ]:
# coefficients for [tenure, MonthlyCharges]: positive pushes towards churn, negative towards staying
sgd_2d.intercept_, sgd_2d.coef_

In [ ]:
from matplotlib.colors import ListedColormap

custom_cmap = ListedColormap(["#9898ff", "#a0faa0"])  # blue = stay, green = churn

# for the contour plot
x0, x1 = np.meshgrid(np.linspace(0, 72, 500).reshape(-1, 1),
                     np.linspace(15, 120, 200).reshape(-1, 1))
X_new = np.c_[x0.ravel(), x1.ravel()]  # one instance per point on the figure
X_new_scaled = scaler.transform(X_new)

y_proba = sgd_2d.predict_proba(X_new_scaled)
y_predict = sgd_2d.predict(X_new_scaled)

zz1 = y_proba[:, 1].reshape(x0.shape)
zz = y_predict.reshape(x0.shape)

X_show, y_show = X_test[:500], y_test[:500]  # first 500 test customers, for readability

plt.figure(figsize=(10, 6))
plt.plot(X_show[y_show == 0, 0], X_show[y_show == 0, 1], "bs", label="Stayed")
plt.plot(X_show[y_show == 1, 0], X_show[y_show == 1, 1], "g^", label="Churned")

plt.contourf(x0, x1, zz, cmap=custom_cmap)
contour = plt.contour(x0, x1, zz1, cmap="hot")
plt.clabel(contour, inline=1)
plt.xlabel("tenure (months)")
plt.ylabel("MonthlyCharges")
plt.legend(loc="lower right")
plt.axis([0, 72, 15, 120])
plt.grid()
plt.show()